# 12 — v12: CatBoost as 4th Base Model in Stacking

**Baseline:** v9/v10 postal-sector OOF smoothing → Kaggle RMSE **21,755.56** | OOF RMSE **~$21,627**  
**Goal:** Add CatBoost as a 4th stacking model alongside XGB + LGBM + ExtraTrees.

## Why CatBoost?

| Property | XGBoost | LightGBM | ExtraTrees | **CatBoost** |
|---|---|---|---|---|
| Tree growth | Level-wise | Leaf-wise | Random split | **Symmetric (oblivious)** |
| Speed | Medium | Fastest | Medium | Slower |
| Overfitting risk | Low | Higher on small data | Low | **Very low** |
| Diversity vs others | — | Medium | High | **High** |

CatBoost's symmetric tree structure makes its predictions structurally different from XGB/LGBM,  
which gives the Ridge meta-learner more diversity to blend — the key driver of stacking gains.

## What changes vs v9

| # | Change | Section |
|---|---|---|
| 1 | Install `catboost` library | 1 |
| 2 | CatBoost HPO (RandomizedSearchCV, 20 trials) | 6d |
| 3 | OOF loop extended to 4 models | 9 |
| 4 | Ridge meta on 4 OOF features (vs 3) | 11 |

All other settings (features, preprocessing, encoding, train/val split) are identical to v9.

---
## 1. Imports & Load Data

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'catboost', '-q'])
print('catboost installed.')

catboost installed.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

Train: (150634, 77)  |  Test: (16737, 76)


---
## 2. Feature Engineering

Identical to v9/v10 — no changes. Features include:
- `postal_sector`, `block_num`, `amenity_score`, `dist_to_cbd`, `street_freq`, etc.

In [4]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)

    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    df['postal_sector'] = df['postal'].astype(str).str[:4]
    df['block_num']     = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)

    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

Train: (150634, 94)  |  Test: (16737, 93)


---
## 3. Per-Fold OOF Target Encoding

Same as v9: `town`, `postal_sector` (min_count=10), `flat_model`.

In [5]:
MIN_COUNT_SECTOR = 10

def oof_group_median(group_series, y_series, n_splits=5, random_state=42, min_count=1):
    """OOF median target encoding. Groups with < min_count rows fall back to global median."""
    groups = group_series.values
    y      = y_series.values
    encoded    = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_count}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

train_fe['town_median_price']          = oof_group_median(train_fe['town'],          train['resale_price'])
train_fe['postal_sector_median_price'] = oof_group_median(train_fe['postal_sector'], train['resale_price'],
                                                           min_count=MIN_COUNT_SECTOR)
train_fe['flat_model_median_price']    = oof_group_median(train_fe['flat_model'],    train['resale_price'])

_tmp = pd.DataFrame({
    'town':          train_fe['town'].values,
    'postal_sector': train_fe['postal_sector'].values,
    'flat_model':    train_fe['flat_model'].values,
    'resale_price':  train['resale_price'].values,
})
full_town_map   = _tmp.groupby('town')['resale_price'].median()
sector_counts   = _tmp.groupby('postal_sector')['resale_price'].count()
valid_sectors   = sector_counts[sector_counts >= MIN_COUNT_SECTOR].index
full_sector_map = _tmp[_tmp['postal_sector'].isin(valid_sectors)].groupby('postal_sector')['resale_price'].median()
full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()

test_fe['town_median_price']          = test_fe['town'].map(full_town_map).fillna(full_town_map.median())
test_fe['postal_sector_median_price'] = test_fe['postal_sector'].map(full_sector_map).fillna(full_sector_map.median())
test_fe['flat_model_median_price']    = test_fe['flat_model'].map(full_model_map).fillna(full_model_map.median())

print(f'OOF encoding done. Rare sectors smoothed: {(sector_counts < MIN_COUNT_SECTOR).sum()} of {len(sector_counts)}')

OOF encoding done. Rare sectors smoothed: 32 of 598


---
## 4. Prepare X and y

In [6]:
DROP_ALL = (['id', 'resale_price', 'postal', 'block']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms'])

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 71  (num=56, cat=15)


---
## 5. Preprocessing Pipeline

In [7]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)
print('Preprocessor ready.')

Preprocessor ready.


---
## 6a. RandomizedSearchCV — XGBoost

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

param_dist_xgb = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [4, 5, 6, 7, 8, 9],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.4, 0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5, 7, 10],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0.5, 1.0, 1.5, 2.0, 3.0],
}
xgb_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))]),
    param_distributions=param_dist_xgb, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
xgb_search.fit(X_train, y_train_log)
print(f'\nXGB best CV RMSE: ${-xgb_search.best_score_:,.0f}')
print('Best params:', xgb_search.best_params_)

XGB_PARAMS = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
XGB_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':0, 'tree_method':'hist'})

xgb_check = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
xgb_check.fit(X_train, y_train_log)
xgb_val_rmse = rmse(y_val, np.expm1(xgb_check.predict(X_val)))
print(f'XGB val RMSE: ${xgb_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

XGB best CV RMSE: $22,656
Best params: {'model__subsample': 0.9, 'model__reg_lambda': 1.5, 'model__reg_alpha': 0.01, 'model__n_estimators': 2000, 'model__min_child_weight': 7, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.4}
XGB val RMSE: $21,763


---
## 6b. RandomizedSearchCV — LightGBM

In [9]:
param_dist_lgbm = {
    'model__n_estimators'     : [500, 800, 1000, 1500, 2000],
    'model__max_depth'        : [5, 6, 7, 8, 10, 12],
    'model__num_leaves'       : [60, 80, 100, 127, 200, 300],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.07, 0.1],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30, 50],
    'model__reg_alpha'        : [0, 0.01, 0.1, 1.0],
    'model__reg_lambda'       : [0, 0.1, 0.5, 1.0],
}
lgbm_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))]),
    param_distributions=param_dist_lgbm, n_iter=40, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
lgbm_search.fit(X_train, y_train_log)
print(f'\nLGBM best CV RMSE: ${-lgbm_search.best_score_:,.0f}')
print('Best params:', lgbm_search.best_params_)

LGBM_PARAMS = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}
LGBM_PARAMS.update({'random_state':42, 'n_jobs':-1, 'verbosity':-1})

lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
lgbm_check.fit(X_train, y_train_log)
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}')

Fitting 3 folds for each of 40 candidates, totalling 120 fits

LGBM best CV RMSE: $22,524
Best params: {'model__subsample': 0.8, 'model__reg_lambda': 0.5, 'model__reg_alpha': 0, 'model__num_leaves': 300, 'model__n_estimators': 1000, 'model__min_child_samples': 20, 'model__max_depth': 12, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.5}
LGBM val RMSE: $21,563


---
## 6c. RandomizedSearchCV — Extra Trees

In [10]:
param_dist_et = {
    'model__n_estimators'     : [200, 300, 500],
    'model__max_depth'        : [15, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf' : [1, 2, 4],
    'model__max_features'     : [0.5, 0.6, 0.7, 0.8],
}
et_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(random_state=42, n_jobs=-1))]),
    param_distributions=param_dist_et, n_iter=15, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
et_search.fit(X_train, y_train_log)
print(f'ET best CV RMSE: ${-et_search.best_score_:,.0f}')
et_val_rmse = rmse(y_val, np.expm1(et_search.best_estimator_.predict(X_val)))
print(f'ET val RMSE:     ${et_val_rmse:,.0f}')

ET_PARAMS = {k.replace('model__',''):v for k,v in et_search.best_params_.items()}
ET_PARAMS.update({'random_state':42,'n_jobs':-1})
print(f'ET_PARAMS: {ET_PARAMS}')

Fitting 3 folds for each of 15 candidates, totalling 45 fits
ET best CV RMSE: $24,462
ET val RMSE:     $23,270
ET_PARAMS: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 20, 'random_state': 42, 'n_jobs': -1}


---
## 6d. RandomizedSearchCV — CatBoost (NEW)

CatBoost uses symmetric (oblivious) trees — each level uses the **same** split across all nodes.  
This acts as a strong regularizer, making CatBoost structurally different from XGB/LGBM.

Key params:
- `iterations` — number of trees (equivalent to `n_estimators`)
- `depth` — tree depth (max 10; symmetric so shallow depths still powerful)
- `l2_leaf_reg` — L2 regularization on leaf weights
- `colsample_bylevel` — fraction of features considered per tree level

In [11]:
param_dist_cat = {
    'model__iterations'        : [500, 800, 1000, 1200],
    'model__depth'             : [4, 5, 6, 7, 8],
    'model__learning_rate'     : [0.03, 0.05, 0.08, 0.1],
    'model__l2_leaf_reg'       : [1, 3, 5, 10],
    'model__colsample_bylevel' : [0.6, 0.7, 0.8, 1.0],
    'model__min_child_samples' : [5, 10, 20],
}
cat_search = RandomizedSearchCV(
    Pipeline([('pre', preprocessor),
              ('model', CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=0))]),
    param_distributions=param_dist_cat, n_iter=20, cv=3,
    scoring=dollar_rmse_scorer, random_state=42, n_jobs=-1, verbose=1, refit=True,
)
cat_search.fit(X_train, y_train_log)
print(f'\nCatBoost best CV RMSE: ${-cat_search.best_score_:,.0f}')
print('Best params:', cat_search.best_params_)

CAT_PARAMS = {k.replace('model__',''):v for k,v in cat_search.best_params_.items()}
CAT_PARAMS.update({'loss_function':'RMSE', 'random_seed':42, 'verbose':0})

cat_check = Pipeline([('pre', preprocessor), ('model', CatBoostRegressor(**CAT_PARAMS))])
cat_check.fit(X_train, y_train_log)
cat_val_rmse = rmse(y_val, np.expm1(cat_check.predict(X_val)))
print(f'CatBoost val RMSE: ${cat_val_rmse:,.0f}')

Fitting 3 folds for each of 20 candidates, totalling 60 fits

CatBoost best CV RMSE: $23,132
Best params: {'model__min_child_samples': 5, 'model__learning_rate': 0.08, 'model__l2_leaf_reg': 1, 'model__iterations': 1000, 'model__depth': 7, 'model__colsample_bylevel': 0.7}
CatBoost val RMSE: $22,500


---
## 7. HPO Summary

Compare val RMSE across all 4 models before stacking.

In [ ]:
print('Val RMSE summary (individual models, before stacking):')
print(f'  XGB:      ${xgb_val_rmse:>10,.0f}')
print(f'  LGBM:     ${lgbm_val_rmse:>10,.0f}')
print(f'  ET:       ${et_val_rmse:>10,.0f}')
print(f'  CatBoost: ${cat_val_rmse:>10,.0f}')
print(f'\nv9 Kaggle RMSE for reference: $21,755')

---
## 8. Generate OOF Predictions (5-Fold) — 4 Models

Per-fold target encoding recomputed inside each fold (no leakage).  
**New:** CatBoost added alongside XGB, LGBM, ExtraTrees.

In [12]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))
cat_oof  = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))
cat_test_folds  = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []
fold_rmses_cat  = []

ENCODE_PAIRS = [
    ('town',          'town_median_price',          1),
    ('postal_sector', 'postal_sector_median_price', MIN_COUNT_SECTOR),
    ('flat_model',    'flat_model_median_price',    1),
]

print('Generating OOF predictions (5-fold, 4 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}  {"CAT RMSE":>12}')
print('-' * 70)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr  = X.iloc[tr_idx].copy()
    X_va  = X.iloc[va_idx].copy()
    fe_tr = train_fe.iloc[tr_idx]  # use train_fe for group cols — safe even if cols are in DROP_ALL
    fe_va = train_fe.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    for group_col, price_col, min_ct in ENCODE_PAIRS:
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(fe_tr[group_col].values, y_tr):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_ct}
        X_tr[price_col] = fe_tr[group_col].map(fold_map).fillna(global_med_tr).values
        X_va[price_col] = fe_va[group_col].map(fold_map).fillna(global_med_tr).values

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    # CatBoost
    cat_pipe = Pipeline([('pre', preprocessor), ('model', CatBoostRegressor(**CAT_PARAMS))])
    cat_pipe.fit(X_tr, y_tr_log)
    cat_oof[va_idx]      = cat_pipe.predict(X_va)
    cat_test_folds[fold] = cat_pipe.predict(X_test)
    fold_rmses_cat.append(rmse(y_va, np.expm1(cat_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}  ${fold_rmses_cat[-1]:>10,.0f}')

print('-' * 70)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}  ${np.mean(fold_rmses_cat):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)
cat_test_avg  = cat_test_folds.mean(axis=0)

print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 4 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE      CAT RMSE
----------------------------------------------------------------------
    1  $    21,623  $    21,468  $    23,227  $    22,404
    2  $    22,414  $    22,283  $    23,949  $    23,166
    3  $    21,696  $    21,657  $    23,519  $    22,660
    4  $    21,868  $    21,876  $    23,681  $    22,639
    5  $    21,870  $    21,702  $    23,563  $    22,661
----------------------------------------------------------------------
 Mean  $    21,894  $    21,797  $    23,588  $    22,706

OOF generation complete.


---
## 9. Sanity Check — Individual Models & Blends

In [13]:
xgb_oof_rmse  = rmse(y, np.expm1(xgb_oof))
lgbm_oof_rmse = rmse(y, np.expm1(lgbm_oof))
et_oof_rmse   = rmse(y, np.expm1(et_oof))
cat_oof_rmse  = rmse(y, np.expm1(cat_oof))

print('Individual OOF RMSE:')
print(f'  XGB:      ${xgb_oof_rmse:>10,.0f}')
print(f'  LGBM:     ${lgbm_oof_rmse:>10,.0f}')
print(f'  ET:       ${et_oof_rmse:>10,.0f}')
print(f'  CatBoost: ${cat_oof_rmse:>10,.0f}')

blend_3 = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
blend_4 = np.expm1((xgb_oof + lgbm_oof + et_oof + cat_oof) / 4)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_3):>8,.0f}')
print(f'4-model equal-weight blend OOF RMSE: ${rmse(y, blend_4):>8,.0f}')
print(f'\nv9 OOF RMSE (3-model baseline): ~$21,627')

Individual OOF RMSE:
  XGB:      $    21,896
  LGBM:     $    21,799
  ET:       $    23,589
  CatBoost: $    22,707

3-model equal-weight blend OOF RMSE: $  21,752
4-model equal-weight blend OOF RMSE: $  21,659

v9 OOF RMSE (3-model baseline): ~$21,627


---
## 10. Ridge Meta-Model on 4 OOF Features

Ridge learns optimal blend weights across all 4 base models.  
All 4 coefficients should be positive — if CatBoost coef is near 0 or negative, it adds no value.

In [14]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof, cat_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg, cat_test_avg])

print('Ridge alpha sweep (4 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB":>8}  {"LGBM":>8}  {"ET":>8}  {"CAT":>8}')
print('-' * 68)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>8.4f}  {ridge.coef_[1]:>8.4f}  {ridge.coef_[2]:>8.4f}  {ridge.coef_[3]:>8.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:      {meta_model.coef_[0]:.4f}')
print(f'  LGBM:     {meta_model.coef_[1]:.4f}')
print(f'  ET:       {meta_model.coef_[2]:.4f}')
print(f'  CatBoost: {meta_model.coef_[3]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE: ${best_meta_rmse:,.0f}')
print(f'v9 OOF baseline:     ~$21,627')
print(f'Improvement:         ${21627 - best_meta_rmse:,.0f}')

Ridge alpha sweep (4 meta-features):
     alpha          RMSE       XGB      LGBM        ET       CAT
--------------------------------------------------------------------
     0.001  $    21,578    0.2937    0.4206    0.1232    0.1656
     0.010  $    21,578    0.2937    0.4205    0.1233    0.1656
     0.100  $    21,578    0.2941    0.4195    0.1236    0.1659
     1.000  $    21,578    0.2969    0.4104    0.1266    0.1692
    10.000  $    21,582    0.3027    0.3588    0.1484    0.1933
   100.000  $    21,616    0.2732    0.2801    0.2096    0.2392

Best Ridge alpha: 0.1
Learned coefficients:
  XGB:      0.2941
  LGBM:     0.4195
  ET:       0.1236
  CatBoost: 0.1659
  Intercept: -0.0398

OOF meta-train RMSE: $21,578
v9 OOF baseline:     ~$21,627
Improvement:         $49


---
## 11. Generate Submission v12

In [15]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v12 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v12 = sub_v12.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v12_catboost.csv'
sub_v12.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v12.shape}')
print(f'Price range: ${sub_v12.Predicted.min():,.0f} – ${sub_v12.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v12.Predicted.mean():,.0f}')

Saved: ../../outputs/submissions/sub_v12_catboost.csv
Shape: (16735, 2)
Price range: $178,136 – $1,168,735
Price mean:  $450,332


---
## 12. Summary

### Score tracker

| Version | Model | OOF RMSE | Kaggle |
|---|---|---|---|
| v8 | XGB+LGBM+ET stack | ~$21,804 | $21,804.67 |
| v9/v10 | + postal-sector OOF smoothing | ~$21,627 | $21,755.56 |
| v11 | + H3 geospatial (r7/r8) | ~$21,625 | (not submitted) |
| **v12** | **+ CatBoost as 4th base model** | **_(run above)_** | **_(submit if improved)_** |

### Interpreting Ridge coefficients

- **All 4 positive**: CatBoost contributes genuine signal — blend is healthy
- **CatBoost coef near 0**: CatBoost is redundant — likely duplicating XGB/LGBM signal
- **CatBoost coef negative**: Rare; suggests CatBoost predictions are anti-correlated with errors — still helps via error correction

### Next steps
- [ ] Submit if OOF RMSE < $21,625 (v11 baseline)
- [ ] Try Optuna for smarter CatBoost HPO (if val RMSE > $22,000)
- [ ] Pipeline refactor + MLflow (saved as pending task)